In [1]:
# ============================================================
# EVALUATION SETUP
# ============================================================

from ultralytics import YOLO
from pathlib import Path
import torch

project_path = Path(r"G:\AIIC")

data_yaml = (
    project_path
    / "yolo_segmentation"
    / "dataset"
    / "data.yaml"
)

best_model_path = (
    project_path
    / "yolo_segmentation"
    / "runs"
    / "yolo11n_seg_aiic"
    / "weights"
    / "best.pt"
)

evaluation_path = (
    project_path
    / "yolo_segmentation"
    / "evaluation"
)

evaluation_path.mkdir(
    parents=True,
    exist_ok=True
)

print("=" * 70)
print("MODEL EVALUATION SETUP")
print("=" * 70)

print("Model :", best_model_path)
print("Data  :", data_yaml)

print("CUDA  :", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU   :", torch.cuda.get_device_name(0))

model = YOLO(str(best_model_path))

print("\n✅ best.pt loaded")

MODEL EVALUATION SETUP
Model : G:\AIIC\yolo_segmentation\runs\yolo11n_seg_aiic\weights\best.pt
Data  : G:\AIIC\yolo_segmentation\dataset\data.yaml
CUDA  : True
GPU   : NVIDIA GeForce RTX 5060

✅ best.pt loaded


In [2]:
# ============================================================
# EVALUATE ON HELD-OUT TEST SET
# ============================================================

test_metrics = model.val(
    data=str(data_yaml),

    split="test",

    imgsz=640,
    batch=8,
    device=0,
    workers=4,

    plots=True,

    project=str(evaluation_path),
    name="test_evaluation",

    exist_ok=True,
    verbose=True
)

print("\n" + "=" * 70)
print("TEST EVALUATION COMPLETE")
print("=" * 70)

Ultralytics 8.4.137  Python-3.14.3 torch-2.13.0+cu130 CUDA:0 (NVIDIA GeForce RTX 5060, 8123MiB)
YOLO11n-seg summary (fused): 114 layers, 2,902,731 parameters, 0 gradients, 10.0 GFLOPs
val: Fast image access  (ping: 0.10.0 ms, read: 63.832.1 MB/s, size: 2145.7 KB)
val: Scanning G:\AIIC\yolo_segmentation\dataset\labels\test... 414 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 414/414 69.9it/s 5.9s0.1ss
val: G:\AIIC\yolo_segmentation\dataset\images\test\E002_Arduino_Nano__WhatsApp Image 2026-08-11 at 6.07.15 PM (1).jpg: 1 duplicate labels removed
val: New cache created: G:\AIIC\yolo_segmentation\dataset\labels\test.cache
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95)     Mask(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 52/52 4.3it/s 12.1s<0.2s
                   all        414        900      0.757      0.703      0.749      0.657      0.737      0.691       0.73      0.622
    C001_tracing paper          3          3       0.61   

In [4]:
# ============================================================
# CLEAN TEST METRICS SUMMARY
# ============================================================

results = test_metrics.results_dict

print("=" * 70)
print("YOLO INSTANCE SEGMENTATION — TEST RESULTS")
print("=" * 70)

print("\nBOUNDING BOX PERFORMANCE")
print("-" * 70)

print(
    f"Precision      : "
    f"{results.get('metrics/precision(B)', 0):.4f}"
)

print(
    f"Recall         : "
    f"{results.get('metrics/recall(B)', 0):.4f}"
)

print(
    f"mAP@50         : "
    f"{results.get('metrics/mAP50(B)', 0):.4f}"
)

print(
    f"mAP@50-95      : "
    f"{results.get('metrics/mAP50-95(B)', 0):.4f}"
)


print("\nSEGMENTATION MASK PERFORMANCE")
print("-" * 70)

print(
    f"Precision      : "
    f"{results.get('metrics/precision(M)', 0):.4f}"
)

print(
    f"Recall         : "
    f"{results.get('metrics/recall(M)', 0):.4f}"
)

print(
    f"mAP@50         : "
    f"{results.get('metrics/mAP50(M)', 0):.4f}"
)

print(
    f"mAP@50-95      : "
    f"{results.get('metrics/mAP50-95(M)', 0):.4f}"
)

print("\n" + "=" * 70)

YOLO INSTANCE SEGMENTATION — TEST RESULTS

BOUNDING BOX PERFORMANCE
----------------------------------------------------------------------
Precision      : 0.7570
Recall         : 0.7032
mAP@50         : 0.7491
mAP@50-95      : 0.6566

SEGMENTATION MASK PERFORMANCE
----------------------------------------------------------------------
Precision      : 0.7373
Recall         : 0.6910
mAP@50         : 0.7296
mAP@50-95      : 0.6222



In [5]:
# ============================================================
# PER-CLASS SEGMENTATION PERFORMANCE
# ============================================================

import pandas as pd

class_names = model.names

box_maps = test_metrics.box.maps
mask_maps = test_metrics.seg.maps

rows = []

for class_id, class_name in class_names.items():

    rows.append({
        "Class ID": class_id,
        "Class": class_name,
        "Box mAP50-95": float(box_maps[class_id]),
        "Mask mAP50-95": float(mask_maps[class_id])
    })

per_class_df = pd.DataFrame(rows)

per_class_df = per_class_df.sort_values(
    by="Mask mAP50-95",
    ascending=False
).reset_index(drop=True)

print("=" * 70)
print("TOP 15 CLASSES — MASK mAP50-95")
print("=" * 70)

display(
    per_class_df.head(15)
)

print("\n" + "=" * 70)
print("BOTTOM 15 CLASSES — MASK mAP50-95")
print("=" * 70)

display(
    per_class_df.tail(15)
)

TOP 15 CLASSES — MASK mAP50-95


,Class ID,Class,Box mAP50-95,Mask mAP50-95
0,1,C002_paper cup,0.995,0.995000
1,8,C009_popsicle stick,0.971,0.995000
2,13,C014_A4 colored paper,0.995,0.995000
3,14,C015_ribbon,0.995,0.995000
4,56,L004_beakers 1000ml,0.945,0.995000
5,77,L025_Lithium Chloride,0.995,0.995000
6,100,T007_CuttingMat,0.951,0.995000
7,64,L012_gloves LN,0.962,0.995000
8,72,L020_Potassium Iodide,0.995,0.995000
9,73,L021_Methylated Spirit Solution 95%,0.995,0.995000



BOTTOM 15 CLASSES — MASK mAP50-95


,Class ID,Class,Box mAP50-95,Mask mAP50-95
94,28,E006_NodeMCU,0.262559,0.264253
95,24,E002_Arduino Nano,0.251524,0.254941
96,85,S007_BinderClip,0.184410,0.235455
97,26,E004_Breadboard,0.246687,0.221040
98,41,E019_LED Blue,0.256907,0.216173
99,23,E001_Arduino Uno,0.175982,0.166326
100,90,S012_Sellotape,0.182461,0.156403
101,70,L018_glass rod,0.258933,0.132675
102,49,E027_Jumper Wire M-F,0.123818,0.112423
103,82,S004_Paper,0.533167,0.108500
